# NM i AI 2026 — Oppgave 1: Object Detection v2
**YOLOv8l + product reference images + imgsz=1280**

In [ ]:
# CELLE 1: Installer pakker
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
!pip install -q ultralytics==8.1.0
!pip show ultralytics | grep Version

In [ ]:
# CELLE 2: Mount Google Drive og pakk ut train.zip
from google.colab import drive
import zipfile
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

print('Pakker ut train.zip...')
with zipfile.ZipFile('/content/drive/MyDrive/train.zip', 'r') as z:
    z.extractall('/content/')

imgs = list(Path('/content/train/images').glob('*.jpg')) + list(Path('/content/train/images').glob('*.jpeg'))
print(f'Treningsbilder: {len(imgs)}')  # skal si 248

In [ ]:
# CELLE 3: Konverter COCO-annotasjoner til YOLO-format
import json
from pathlib import Path

with open('/content/train/annotations.json') as f:
    coco = json.load(f)

print(f'Bilder: {len(coco["images"])}, Kategorier: {len(coco["categories"])}, Annotasjoner: {len(coco["annotations"])}')

labels_dir = Path('/content/train/labels')
labels_dir.mkdir(exist_ok=True)

image_info = {img['id']: img for img in coco['images']}
annotations_by_image = {}
for ann in coco['annotations']:
    annotations_by_image.setdefault(ann['image_id'], []).append(ann)

for img_id, img in image_info.items():
    W, H = img['width'], img['height']
    stem = Path(img['file_name']).stem
    lines = []
    for ann in annotations_by_image.get(img_id, []):
        x, y, w, h = ann['bbox']
        xc = max(0.0, min(1.0, (x + w / 2) / W))
        yc = max(0.0, min(1.0, (y + h / 2) / H))
        wn = max(0.0, min(1.0, w / W))
        hn = max(0.0, min(1.0, h / H))
        lines.append(f'{ann["category_id"]} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}')
    (labels_dir / f'{stem}.txt').write_text('\n'.join(lines))

print(f'Konverterte {len(image_info)} label-filer')

In [ ]:
# CELLE 4: Pakk ut product images og lag treningseksempler
import json
import zipfile
import cv2
from pathlib import Path

print('Pakker ut product_images.zip...')
with zipfile.ZipFile('/content/drive/MyDrive/product_images.zip', 'r') as z:
    z.extractall('/content/')
print('Ferdig!')

with open('/content/train/annotations.json') as f:
    coco = json.load(f)
with open('/content/NM_NGD_product_images/metadata.json') as f:
    meta = json.load(f)

# Match produktnavn -> category_id
cat_name_to_id = {c['name'].strip().upper(): c['id'] for c in coco['categories']}
code_to_cat = {}
for p in meta['products']:
    name = p['product_name'].strip().upper()
    if name in cat_name_to_id:
        code_to_cat[p['product_code']] = cat_name_to_id[name]
print(f'Matchet {len(code_to_cat)}/329 produkter')

# Lag YOLO-labels for hvert produktbilde (full bilde = senterpunkt 0.5, 0.5, størrelse 1.0, 1.0)
product_labels_dir = Path('/content/product_labels')
product_labels_dir.mkdir(exist_ok=True)
product_images_list = []

img_base = Path('/content/NM_NGD_product_images')
for product_code, cat_id in code_to_cat.items():
    product_dir = img_base / product_code
    if not product_dir.exists():
        continue
    for img_path in product_dir.glob('*.jpg'):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        label_name = f'{product_code}_{img_path.stem}.txt'
        (product_labels_dir / label_name).write_text(f'{cat_id} 0.5 0.5 1.0 1.0')
        product_images_list.append(str(img_path))

print(f'Laget {len(product_images_list)} product image treningseksempler')

In [ ]:
# CELLE 5: Lag kombinert dataset.yaml (hyllebilder + product images)
import random
import json
from pathlib import Path

all_shelf = sorted(Path('/content/train/images').glob('*.jpg')) + \
            sorted(Path('/content/train/images').glob('*.jpeg'))
random.seed(42)
random.shuffle(all_shelf)
split = int(len(all_shelf) * 0.9)
train_shelf = all_shelf[:split]
val_shelf = all_shelf[split:]

# Train: hyllebilder + product images. Val: kun hyllebilder (realistisk eval)
train_all = [str(p) for p in train_shelf] + product_images_list
random.shuffle(train_all)

Path('/content/train_list.txt').write_text('\n'.join(train_all))
Path('/content/val_list.txt').write_text('\n'.join(str(p) for p in val_shelf))
print(f'Train: {len(train_all)} ({len(train_shelf)} hylle + {len(product_images_list)} produkt)')
print(f'Val: {len(val_shelf)}')

with open('/content/train/annotations.json') as f:
    coco = json.load(f)
cats = sorted(coco['categories'], key=lambda c: c['id'])
yaml_lines = ['path: /content', 'train: train_list.txt', 'val: val_list.txt', '',
              f'nc: {len(cats)}', 'names:']
for c in cats:
    safe = c['name'].replace("'", "").replace('"', '')
    yaml_lines.append(f"  {c['id']}: '{safe}'")
Path('/content/dataset.yaml').write_text('\n'.join(yaml_lines))
print('dataset.yaml lagret!')

In [ ]:
# CELLE 6: Tren YOLOv8l med imgsz=1280
import os
import torch
import functools
os.environ['WANDB_DISABLED'] = 'true'

_orig = torch.load
torch.load = functools.partial(_orig, weights_only=False)

from ultralytics import YOLO
model = YOLO('yolov8l.pt')

model.train(
    data='/content/dataset.yaml',
    epochs=100,
    imgsz=1280,
    batch=8,
    device=0,
    half=True,
    patience=20,
    name='nmiai_shelf_v2',
    project='runs',
    exist_ok=True,
)
print('Trening ferdig!')
print('Best model: runs/nmiai_shelf_v2/weights/best.pt')

In [ ]:
# CELLE 7: Eksporter til ONNX (unngår torch-versjonsmismatch i sandbox)
!pip install -q onnxscript

import torch, functools
_orig = torch.load
torch.load = functools.partial(_orig, weights_only=False)

from ultralytics import YOLO
model = YOLO('runs/nmiai_shelf_v2/weights/best.pt')
model.export(format='onnx', opset=17, half=False, imgsz=1280)
print('ONNX eksportert: runs/nmiai_shelf_v2/weights/best.onnx')

In [ ]:
# CELLE 8: Lag run.py, ZIP og last ned
run_py = '''
import argparse
import json
from pathlib import Path
from ultralytics import YOLO


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument("--output", required=True)
    args = parser.parse_args()

    input_dir = Path(args.input)
    output_path = Path(args.output)

    model_path = Path(__file__).parent / "best.onnx"
    model = YOLO(str(model_path))

    predictions = []
    img_paths = sorted(input_dir.glob("*.jpg")) + sorted(input_dir.glob("*.jpeg"))
    print(f"Processing {len(img_paths)} images...")

    for img_path in img_paths:
        image_id = int(img_path.stem.split("_")[-1])
        results = model.predict(
            str(img_path),
            device=0,
            conf=0.20,
            iou=0.45,
            verbose=False,
            imgsz=1280,
        )
        for result in results:
            boxes = result.boxes
            for i in range(len(boxes)):
                x1, y1, x2, y2 = boxes.xyxy[i].tolist()
                predictions.append({
                    "image_id": image_id,
                    "category_id": int(boxes.cls[i].item()),
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "score": float(boxes.conf[i].item()),
                })

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(predictions, f)
    print(f"Wrote {len(predictions)} predictions to {output_path}")


if __name__ == "__main__":
    main()
'''.strip()

import zipfile
from pathlib import Path

Path('/content/submission/run.py').parent.mkdir(exist_ok=True)
Path('/content/submission/run.py').write_text(run_py)

zip_path = '/content/submission_mathea_v2.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write('/content/submission/run.py', 'run.py')
    zf.write('runs/nmiai_shelf_v2/weights/best.onnx', 'best.onnx')

with zipfile.ZipFile(zip_path, 'r') as zf:
    for info in zf.infolist():
        print(f'  {info.filename}  ({info.file_size/1024/1024:.1f} MB)')

zip_mb = Path(zip_path).stat().st_size / 1024 / 1024
print(f'Total ZIP: {zip_mb:.1f} MB (maks 420 MB)')

from google.colab import files
files.download(zip_path)
print('Last opp submission_mathea_v2.zip på app.ainm.no/submit/norgesgruppen-data')